# DepositIQ 01 — Data Understanding and Quality Profile

**Rubric:** Criterion 2 — Dataset & Data Understanding · **Task:** T11 · **Phase:** 2
**Primary dataset:** `data/raw/bank-full.csv` (45,211 × 17, decision D20)
**Companion document:** [docs/02_data_dictionary.md](../docs/02_data_dictionary.md) (task T10)

## What this notebook is for

T10 documented *what each column means*. This notebook establishes **what the data is actually
like**: whether it is complete, consistent, duplicated, degenerate or skewed, with the evidence
printed rather than described.

Sections: (1) inventory, (2) structure and dtypes, (3) true nulls, (4) exact duplicate rows,
(5) constant / near-constant columns, (6) `unknown` as disguised missingness, (7) the `pdays`
sentinel and whether it can be trusted, (8) skew and outliers, (9) negative values, (10) saved
figures, (11) findings, (12) **independent cross-check against the data dictionary**.

## The rule this notebook follows

Section 12 **re-derives every figure that `docs/02_data_dictionary.md` quotes and compares the two
lists**. A mismatch is printed as a ⚠ DISAGREEMENT and a documented *meaning* that the data refines
is printed as a NUANCE — in both cases the dictionary is left exactly as written. T11 measures and
reports; it does not overwrite another task's deliverable.

That check is not decoration. In T10 exactly this kind of independent re-measurement caught a wrong
column classification (I had `balance` as shared and `month` as classic-only; the data says the
opposite), and on this notebook's first run it caught six defects **in this notebook's own parser**
rather than real drift — a failure mode worth naming, because a parser bug and a genuine data
change look identical from the inside.

## How to run

From the repository root, inside the project virtual environment:

```bash
jupyter lab notebooks/01_data_understanding.ipynb
# or non-interactively, writing the results back into the file:
jupyter nbconvert --to notebook --execute --inplace notebooks/01_data_understanding.ipynb
```

The setup cell finds the repository root by walking up from the kernel's working directory, so the
notebook behaves the same whether the kernel started in the repo root or in `notebooks/`
(decision D12). Every claim below is produced by the cell that follows it.

## Setup

Imports, and the path handling that makes `from src import ...` work from any working directory.

In [1]:
# Make `from src import ...` work no matter where the kernel was started (decision D12).
# The repository root is found by walking up from the working directory until src/config.py
# appears - nothing here is an absolute path, so the notebook survives being cloned elsewhere.
import platform
import re
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config.py").exists():
            return candidate
    raise RuntimeError("Could not find the repository root above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config          # noqa: E402  (import after sys.path is fixed, as intended)
from src import data_loader as load  # noqa: E402

config.ensure_dirs()            # create reports/figures/ if this is a fresh clone

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

print("Executed       :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python         :", platform.python_version())
print("pandas / numpy :", pd.__version__, "/", np.__version__)
print("matplotlib/sns :", __import__("matplotlib").__version__, "/", sns.__version__)
print("Repo root      :", REPO_ROOT)
print("Figures folder :", config.FIGURES_DIR)

Executed       : 2026-10-27 18:31:01
Python         : 3.11.9
pandas / numpy : 2.3.3 / 2.3.3
matplotlib/sns : 3.11.0 / 0.13.2
Repo root      : D:\diabetes+130-us+hospitals+for+years+1999-2008\depositiq-term-deposit-prediction
Figures folder : D:\diabetes+130-us+hospitals+for+years+1999-2008\depositiq-term-deposit-prediction\reports\figures


## 1. Inventory — do the files match what the plan says?

`src/data_loader.py` is the only place that reads the raw CSVs (decision D13), so this is the
loader's own view of what exists.

In [2]:
inventory = load.variant_report()
print("Datasets exposed by src/data_loader.py, measured now:")
print(inventory.to_string(index=False))
print()

# Expected shapes come from the task plan and the T10 data dictionary. They are CHECKED, not
# assumed: a mismatch prints MISMATCH rather than being quietly absorbed.
EXPECTED_SHAPES = {"full": (45211, 17), "sample": (4521, 17), "additional": (41188, 21)}
for _, row in inventory.iterrows():
    expected = EXPECTED_SHAPES[row["variant"]]
    measured = (int(row["rows"]), int(row["columns"]))
    flag = "OK" if measured == expected else "MISMATCH"
    print(f"  {row['variant']:<11} measured {measured[0]:>6,} x {measured[1]:<3}"
          f" expected {expected[0]:>6,} x {expected[1]:<3} -> {flag}")
print()

# Class balance straight from the loader's (X, y) split - the numbers every later metric depends on.
for name in ("full", "additional"):
    X, y = load.load_xy(name, verbose=False)
    print(f"{name:<11} X = {X.shape[0]:,} x {X.shape[1]:<3} features | "
          f"y: {int((y == 0).sum()):,} no / {int((y == 1).sum()):,} yes = {y.mean():.4%} positive"
          f" | 'duration' still in X: {'duration' in X.columns}")
print()
print("Note: 'duration' is deliberately still present. The loader does not drop it (decision D14)")
print("      because Phase 5 has to DEMONSTRATE the leak before removing it for the main model.")
print("      The additional 10% sample (bank-additional.csv) is not exposed as a loader variant.")

Datasets exposed by src/data_loader.py, measured now:
   variant                     file  rows  columns  features  positive_rate  has_economic_cols
additional bank-additional-full.csv 41188       21        20         0.1127               True
      full            bank-full.csv 45211       17        16         0.1170              False
    sample                 bank.csv  4521       17        16         0.1152              False

  additional  measured 41,188 x 21  expected 41,188 x 21  -> OK
  full        measured 45,211 x 17  expected 45,211 x 17  -> OK
  sample      measured  4,521 x 17  expected  4,521 x 17  -> OK

full        X = 45,211 x 16  features | y: 39,922 no / 5,289 yes = 11.6985% positive | 'duration' still in X: True


additional  X = 41,188 x 20  features | y: 36,548 no / 4,640 yes = 11.2654% positive | 'duration' still in X: True

Note: 'duration' is deliberately still present. The loader does not drop it (decision D14)
      because Phase 5 has to DEMONSTRATE the leak before removing it for the main model.
      The additional 10% sample (bank-additional.csv) is not exposed as a loader variant.


## 2. Structure, dtypes and cardinality

Every column, with the dtype pandas inferred, the number of distinct values, and the numeric
range where one exists. The dtype mix matters: a numeric column that arrived as `object` would
mean unparsed text, and the mean-based profiling in section 8 would silently skip it.

In [3]:
RAW = {name: load.load_raw(name, verbose=False) for name in ("full", "additional", "sample")}


def structure_table(df):
    # One row per column: measured dtype, distinct-value count, numeric range, first value as a sample.
    rows = []
    for column in df.columns:
        series = df[column]
        is_numeric = pd.api.types.is_numeric_dtype(series)
        rows.append({
            "column": column,
            "dtype": str(series.dtype),
            "n_unique": int(series.nunique()),
            "min": series.min() if is_numeric else "",
            "max": series.max() if is_numeric else "",
            "first_value": series.iloc[0],
        })
    return pd.DataFrame(rows)


for name in ("full", "additional"):
    df = RAW[name]
    mix = df.dtypes.astype(str).value_counts().to_dict()
    print("=" * 100)
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns | dtype mix: {mix}")
    print(structure_table(df).to_string(index=False))
    print()

# Parsing sanity check: only the three dtypes we expect may appear anywhere.
allowed = {"int64", "float64", "object"}
for name in ("full", "additional"):
    unexpected = sorted({dtype for dtype in RAW[name].dtypes.astype(str) if dtype not in allowed})
    print(f"{name:<11} dtypes outside {sorted(allowed)}: {unexpected if unexpected else 'none'}")

full: 45,211 rows x 17 columns | dtype mix: {'object': 10, 'int64': 7}
   column  dtype  n_unique   min    max first_value
      age  int64        77    18     95          58
      job object        12               management
  marital object         3                  married
education object         4                 tertiary
  default object         2                       no
  balance  int64      7168 -8019 102127        2143
  housing object         2                      yes
     loan object         2                       no
  contact object         3                  unknown
      day  int64        31     1     31           5
    month object        12                      may
 duration  int64      1573     0   4918         261
 campaign  int64        48     1     63           1
    pdays  int64       559    -1    871          -1
 previous  int64        41     0    275           0
 poutcome object         4                  unknown
        y object         2                   

## 3. True null counts — and why they are a misleading measure here

`df.isna()` is the standard first quality check. This notebook reports it, and then shows why it
is close to worthless on this dataset.

In [4]:
for name in ("full", "additional"):
    na = RAW[name].isna().sum()
    print(f"{name:<11} {RAW[name].shape[1]:>2} columns | {int(na.sum()):>3} NaN cells | "
          f"columns containing a NaN: {int((na > 0).sum())}")

total_nan = int(RAW["full"].isna().sum().sum() + RAW["additional"].isna().sum().sum())
print()
print(f"Total NaN cells across both full variants: {total_nan}")
print()
print("Insight 1: isnull() says the data is spotless, and that is exactly the trap. The missingness")
print("           is stored as the literal string 'unknown' (section 6) and as sentinel values in")
print("           'pdays' (section 7), so a null count alone proves nothing about data quality.")
print("           Reporting '0 nulls' as a quality result would be actively misleading.")

full        17 columns |   0 NaN cells | columns containing a NaN: 0
additional  21 columns |   0 NaN cells | columns containing a NaN: 0



Total NaN cells across both full variants: 0

Insight 1: isnull() says the data is spotless, and that is exactly the trap. The missingness
           is stored as the literal string 'unknown' (section 6) and as sentinel values in
           'pdays' (section 7), so a null count alone proves nothing about data quality.
           Reporting '0 nulls' as a quality result would be actively misleading.


## 4. Exact duplicate rows

Duplicate detection is done on **all** columns including the target, so a "duplicate" here means
two byte-identical records — the only sense in which duplication is unambiguous.

In [5]:
for name in ("full", "additional", "sample"):
    df = RAW[name]
    repeats = df.duplicated(keep="first")
    count = int(repeats.sum())
    distinct = len(df.drop_duplicates())
    print(f"{name:<11} rows={len(df):>6,} | duplicate rows={count:>3} ({count / len(df):.4%}) | "
          f"distinct rows={distinct:>6,} | removed by drop_duplicates={len(df) - distinct:>3}")

print()
duplicate_rows = RAW["additional"][RAW["additional"].duplicated(keep="first")]
print(f"Evidence: the {len(duplicate_rows)} repeated rows in the additional variant, shown on a")
print("subset of columns for readability - the duplicate test itself used all 21 columns.")
print(duplicate_rows[["age", "job", "marital", "education", "default", "contact", "month",
                      "day_of_week", "duration", "campaign", "pdays", "previous", "poutcome",
                      "y"]].to_string(index=False))
print()
print("Insight 2: the primary dataset has no duplicate rows at all, so the main model is unaffected.")
print("           The additional variant repeats 12 rows (0.03%). They are left untouched in")
print("           data/raw/; dropping or keeping them is decided and reported explicitly in T14.")

full        rows=45,211 | duplicate rows=  0 (0.0000%) | distinct rows=45,211 | removed by drop_duplicates=  0


additional  rows=41,188 | duplicate rows= 12 (0.0291%) | distinct rows=41,176 | removed by drop_duplicates= 12
sample      rows= 4,521 | duplicate rows=  0 (0.0000%) | distinct rows= 4,521 | removed by drop_duplicates=  0

Evidence: the 12 repeated rows in the additional variant, shown on a
subset of columns for readability - the duplicate test itself used all 21 columns.
 age         job  marital           education default   contact month day_of_week  duration  campaign  pdays  previous    poutcome   y
  39 blue-collar  married            basic.6y      no telephone   may         thu       124         1    999         0 nonexistent  no
  36     retired  married             unknown      no telephone   jul         thu        88         1    999         0 nonexistent  no
  27  technician   single professional.course      no  cellular   jul         mon       331         2    999         0 nonexistent  no
  47  technician divorced         high.school      no  cellular   jul         thu    

## 5. Constant and near-constant columns

A constant column carries no information and should be dropped; a near-constant one carries very
little and will produce unstable importance scores. Both are worth knowing before Phase 5.

In [6]:
def concentration_table(df):
    # How concentrated is each column? share = fraction of rows holding the most common value.
    rows = []
    for column in df.columns:
        shares = df[column].value_counts(normalize=True, dropna=False)
        rows.append({
            "column": column,
            "n_unique": int(df[column].nunique()),
            "most_common_value": shares.index[0],
            "share": float(shares.iloc[0]),
        })
    return pd.DataFrame(rows).sort_values("share", ascending=False).reset_index(drop=True)


concentration = {name: concentration_table(RAW[name]) for name in ("full", "additional")}
for name, table in concentration.items():
    numeric_columns = RAW[name].select_dtypes(include="number")
    low_cardinality = [c for c in numeric_columns.columns if numeric_columns[c].nunique() <= 15]
    print("=" * 100)
    print(f"{name}: 6 most concentrated columns")
    print(table.head(6).to_string(index=False, formatters={"share": "{:.1%}".format}))
    print(f"{name}: numeric columns with <= 15 distinct values -> {low_cardinality}")
    print()

for name, table in concentration.items():
    constant = table[table["n_unique"] == 1]
    high = table[table["share"] >= 0.90]
    print(f"  {name:<11} constant columns: {len(constant)} | "
          f"near-constant (one value >= 90%): {[(row.column, f'{row.share:.1%}') for row in high.itertuples()]}")
print()
print("Insight 3: no column is constant in either variant, so nothing has to be dropped for being")
print("           information-free. Concentration is still very uneven: in the primary set 'default'")
print("           is 98.2% 'no' (only 814 clients are in default), and in the additional set 'pdays'")
print("           is 96.3% '999'. The five economic indicators are also low-cardinality numbers -")
print("           'emp.var.rate' has 10 distinct values and 'nr.employed' 11 across 41,188 rows -")
print("           so they behave more like ordered categories than continuous measurements.")

full: 6 most concentrated columns
  column  n_unique most_common_value share
 default         2                no 98.2%
       y         2                no 88.3%
    loan         2                no 84.0%
poutcome         4           unknown 81.7%
previous        41                 0 81.7%
   pdays       559                -1 81.7%
full: numeric columns with <= 15 distinct values -> []

additional: 6 most concentrated columns
  column  n_unique most_common_value share
   pdays        27               999 96.3%
       y         2                no 88.7%
previous         8                 0 86.3%
poutcome         3       nonexistent 86.3%
    loan         3                no 82.4%
 default         3                no 79.1%
additional: numeric columns with <= 15 distinct values -> ['previous', 'emp.var.rate', 'nr.employed']

  full        constant columns: 0 | near-constant (one value >= 90%): [('default', '98.2%')]
  additional  constant columns: 0 | near-constant (one value >= 90%): [(

## 6. `unknown` — missingness stored as a category

The supplied documentation states: *"There are several missing values in some categorical
attributes, all coded with the 'unknown' label."* So these are genuine missing values that
`isna()` cannot see. The treatment policy is decided in T12; this section only establishes the
facts.

In [7]:
def unknown_profile(df):
    rows = []
    for column in df.columns:
        if df[column].dtype == object:
            count = int((df[column] == "unknown").sum())
            rows.append({
                "column": column,
                "n_levels": int(df[column].nunique()),
                "unknown": count,
                "pct_of_rows": count / len(df),
            })
    return pd.DataFrame(rows).sort_values("unknown", ascending=False).reset_index(drop=True)


unknown = {name: unknown_profile(RAW[name]) for name in ("full", "additional")}
for name, table in unknown.items():
    print("=" * 100)
    print(f"{name}: {len(RAW[name]):,} rows | {len(table)} categorical columns | "
          f"{int((table['unknown'] > 0).sum())} contain an 'unknown' level")
    print(table[table["unknown"] > 0].to_string(
        index=False, formatters={"pct_of_rows": "{:.1%}".format}))
    categorical_cells = len(RAW[name]) * len(table)
    print(f"  'unknown' cells among categorical columns: {int(table['unknown'].sum()):,} "
          f"({table['unknown'].sum() / categorical_cells:.2%} of categorical cells)")
    print()

# The levels differ between variants, so the level lists are printed for the columns that differ.
for column in ("poutcome", "contact", "default", "education", "marital", "job", "housing", "loan"):
    full_unknown = int((RAW["full"][column] == "unknown").sum())
    additional_unknown = int((RAW["additional"][column] == "unknown").sum())
    print(f"  {column:<11} unknown: full={full_unknown:>6,} | additional={additional_unknown:>6,}")

print()
for column in ("poutcome", "contact", "default", "education"):
    print(f"  levels of '{column}':")
    print(f"     full       -> {sorted(RAW['full'][column].unique())}")
    print(f"     additional -> {sorted(RAW['additional'][column].unique())}")

print()
biggest_full = unknown["full"].iloc[0]
biggest_additional = unknown["additional"].iloc[0]
print(f"Insight 4: the largest hidden-missingness block is '{biggest_full['column']}' in the classic")
print(f"           set ({int(biggest_full['unknown']):,} rows = {biggest_full['pct_of_rows']:.1%}) and "
      f"'{biggest_additional['column']}' in the")
print(f"           additional set ({int(biggest_additional['unknown']):,} = "
      f"{biggest_additional['pct_of_rows']:.1%}). Both are far too large to ignore.")
print("           'unknown' is NOT a null: dropping nulls would delete none of it, while listwise")
print("           deletion of 'unknown' would throw away most of the classic set.")
print("           The variants also encode the same idea differently - the classic set's")
print("           'poutcome = unknown' is the additional set's 'poutcome = nonexistent' - so they")
print("           cannot share one encoder (decision D19).")


full: 45,211 rows | 10 categorical columns | 4 contain an 'unknown' level
   column  n_levels  unknown pct_of_rows
 poutcome         4    36959       81.7%
  contact         3    13020       28.8%
education         4     1857        4.1%
      job        12      288        0.6%
  'unknown' cells among categorical columns: 52,124 (11.53% of categorical cells)

additional: 41,188 rows | 11 categorical columns | 6 contain an 'unknown' level
   column  n_levels  unknown pct_of_rows
  default         3     8597       20.9%
education         8     1731        4.2%
  housing         3      990        2.4%
     loan         3      990        2.4%
      job        12      330        0.8%
  marital         4       80        0.2%
  'unknown' cells among categorical columns: 12,718 (2.81% of categorical cells)

  poutcome    unknown: full=36,959 | additional=     0
  contact     unknown: full=13,020 | additional=     0
  default     unknown: full=     0 | additional= 8,597
  education   unknown: 

## 7. `pdays` — a sentinel-encoded numeric column

`pdays` looks like a normal integer. It is not: one specific value means "this client was never
contacted before", and **the sentinel differs between the two variants**. Treating it as a number
produces a mean that averages a flag together with real day counts.

In [8]:
pdays_sentinels = {}
for name in ("full", "additional"):
    series = RAW[name]["pdays"]
    sentinel = int(series.value_counts().index[0])
    pdays_sentinels[name] = sentinel
    never_contacted = series == sentinel
    real_values = series[~never_contacted]
    print("=" * 100)
    print(f"{name}: pdays min={series.min()} max={series.max()} "
          f"mean={series.mean():.1f} median={series.median()}")
    print(f"  sentinel (most frequent value) = {sentinel} -> {int(never_contacted.sum()):,} rows "
          f"({never_contacted.mean():.2%} of this variant)")
    print(f"  the other {len(real_values):,} rows: min={real_values.min()} max={real_values.max()} "
          f"mean={real_values.mean():.1f} median={real_values.median():.0f}")
    print()

print(f"Sentinels measured: {pdays_sentinels}")
print()
print(f"Insight 5: pdays is bimodal in both variants. In the classic set {pdays_sentinels['full']} means")
print(f"           'never contacted before' and covers {int((RAW['full']['pdays'] == pdays_sentinels['full']).sum()):,} rows; the additional set")
print(f"           uses {pdays_sentinels['additional']} for the same meaning. The published means "
      f"({RAW['full']['pdays'].mean():.1f} and")
print(f"           {RAW['additional']['pdays'].mean():.1f}) are therefore meaningless - averaging a flag with real day counts. The fix (T12)")
print("           is a boolean previously_contacted flag plus, where meaningful, a cleaned")
print("           days-since-contact value. Critically, code that hard-codes the classic -1 would")
print("           silently corrupt the additional variant, which is why D19 requires separate")
print("           pipelines per variant. Section 7.1 tests whether the sentinel can be trusted as")
print("           that flag at all.")

full: pdays min=-1 max=871 mean=40.2 median=-1.0
  sentinel (most frequent value) = -1 -> 36,954 rows (81.74% of this variant)
  the other 8,257 rows: min=1 max=871 mean=224.6 median=194

additional: pdays min=0 max=999 mean=962.5 median=999.0
  sentinel (most frequent value) = 999 -> 39,673 rows (96.32% of this variant)
  the other 1,515 rows: min=0 max=27 mean=6.0 median=6

Sentinels measured: {'full': -1, 'additional': 999}

Insight 5: pdays is bimodal in both variants. In the classic set -1 means
           'never contacted before' and covers 36,954 rows; the additional set
           uses 999 for the same meaning. The published means (40.2 and
           962.5) are therefore meaningless - averaging a flag with real day counts. The fix (T12)
           is a boolean previously_contacted flag plus, where meaningful, a cleaned
           days-since-contact value. Critically, code that hard-codes the classic -1 would
           silently corrupt the additional variant, which is why D19 

### 7.1 Can the sentinel be trusted as a "never contacted" flag?

The documentation defines the sentinel as "client was not previously contacted". That claim is
testable: `previous` counts the contacts made before this campaign, so it is an **independent
witness** to the same fact. If a row carries the sentinel *and* `previous > 0`, the two columns
disagree about whether the client had been contacted.

In [9]:
for name in ("full", "additional"):
    df = RAW[name]
    sentinel = pdays_sentinels[name]
    is_sentinel = df["pdays"] == sentinel
    crosstab = pd.crosstab(is_sentinel, df["previous"] > 0)
    crosstab.index = [f"pdays != {sentinel}", f"pdays == {sentinel}"]
    crosstab.columns = ["previous == 0", "previous > 0"]
    contradictory = int((is_sentinel & (df["previous"] > 0)).sum())
    other_level = "unknown" if name == "full" else "nonexistent"

    print("=" * 100)
    print(f"--- {name} (sentinel {sentinel}) ---")
    print(crosstab.to_string())
    print(f"  sentinel rows                     : {int(is_sentinel.sum()):,}")
    print(f"  of those, 'previous > 0'          : {contradictory:,} "
          f"({contradictory / int(is_sentinel.sum()):.2%} of the sentinel rows)")
    print("  independent markers of prior contact, all three measuring the same idea:")
    print(f"     previous > 0            : {int((df['previous'] > 0).sum()):,} rows")
    print(f"     pdays != sentinel       : {int((~is_sentinel).sum()):,} rows")
    print(f"     poutcome != {other_level:<12}: {int((df['poutcome'] != other_level).sum()):,} rows")
    print()

print("Insight 6: the two variants behave differently, and only one of them is self-consistent.")
print("           In the primary set the sentinel is reliable: 'pdays != -1' and 'previous > 0'")
print(f"           select exactly the same {int((RAW['full']['previous'] > 0).sum()):,} rows, with zero contradictions.")
print(f"           In the additional set it is NOT: {int(((RAW['additional']['pdays'] == 999) & (RAW['additional']['previous'] > 0)).sum()):,} rows carry the 999 sentinel")
print("           while 'previous' runs 1-6, and in every one of them 'poutcome' is 'failure'.")
print("           So 999 does not mean 'never contacted' there - it also covers clients whose")
print("           previous campaign failed. Consequence for T12 (decision D21): derive the")
print("           previously_contacted flag from 'previous > 0', which agrees exactly with")
print("           'poutcome != nonexistent' in both variants, instead of from the sentinel. Using")
print("           the sentinel would mislabel 4,110 clients as never contacted and would give them")
print("           a wrong days-since-contact value.")
print("           This is the class of defect no null count, dtype check or duplicate test can see:")
print("           every value here is a valid integer and nothing is missing.")

--- full (sentinel -1) ---
             previous == 0  previous > 0
pdays != -1              0          8257
pdays == -1          36954             0
  sentinel rows                     : 36,954
  of those, 'previous > 0'          : 0 (0.00% of the sentinel rows)
  independent markers of prior contact, all three measuring the same idea:
     previous > 0            : 8,257 rows
     pdays != sentinel       : 8,257 rows
     poutcome != unknown     : 8,252 rows

--- additional (sentinel 999) ---
              previous == 0  previous > 0
pdays != 999              0          1515
pdays == 999          35563          4110
  sentinel rows                     : 39,673
  of those, 'previous > 0'          : 4,110 (10.36% of the sentinel rows)
  independent markers of prior contact, all three measuring the same idea:
     previous > 0            : 5,625 rows
     pdays != sentinel       : 1,515 rows
     poutcome != nonexistent : 5,625 rows

Insight 6: the two variants behave differently, and o

## 8. Numeric spread, skew and IQR-based outliers

For each numeric column: mean, median, standard deviation, skew, excess kurtosis, range, and the
count of values beyond 1.5 × IQR from the quartiles.

In [10]:
def numeric_profile(df):
    rows = []
    for column in df.select_dtypes(include="number").columns:
        series = df[column]
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outliers = int(((series < lower) | (series > upper)).sum())
        rows.append({
            "column": column,
            "mean": series.mean(),
            "median": series.median(),
            "std": series.std(),
            "skew": series.skew(),
            "excess_kurtosis": series.kurtosis(),
            "min": series.min(),
            "max": series.max(),
            "iqr_outliers": outliers,
            "outlier_pct": outliers / len(series),
        })
    return pd.DataFrame(rows)


numeric = {name: numeric_profile(RAW[name]) for name in ("full", "additional")}
PROFILE_FORMATS = {
    "mean": "{:,.1f}".format,
    "median": "{:,.1f}".format,
    "std": "{:,.1f}".format,
    "skew": "{:.2f}".format,
    "excess_kurtosis": "{:.2f}".format,
    "min": "{:g}".format,
    "max": "{:g}".format,
    "outlier_pct": "{:.2%}".format,
}
for name, table in numeric.items():
    print("=" * 100)
    print(f"{name}: {len(table)} numeric columns")
    print(table.to_string(index=False, formatters=PROFILE_FORMATS))
    print()

for name, table in numeric.items():
    ranked = table.reindex(table["skew"].abs().sort_values(ascending=False).index)
    summary = ", ".join(
        f"{row.column} (skew {row.skew:.2f}, {row.iqr_outliers:,} outliers)"
        for row in ranked.head(3).itertuples()
    )
    print(f"  most skewed in {name:<11}: {summary}")
print()
print("Insight 7: the client-level money and campaign columns are strongly right-skewed - 'balance'")
print("           most of all - and the outlier counts are large in absolute terms while being a")
print("           small share of rows. That is the signature of a heavy tail, not of corruption, so")
print("           Phase 3 (T13) has to choose between keeping the values behind a robust scaler and")
print("           winsorising them, and justify the choice: removing them would remove real, mostly")
print("           wealthy clients, who are the ones most able to subscribe.")


full: 7 numeric columns
  column    mean median     std  skew excess_kurtosis   min    max  iqr_outliers outlier_pct
     age    40.9   39.0    10.6  0.68            0.32    18     95           487       1.08%
 balance 1,362.3  448.0 3,044.8  8.36          140.75 -8019 102127          4729      10.46%
     day    15.8   16.0     8.3  0.09           -1.06     1     31             0       0.00%
duration   258.2  180.0   257.5  3.14           18.15     0   4918          3235       7.16%
campaign     2.8    2.0     3.1  4.90           39.25     1     63          3064       6.78%
   pdays    40.2   -1.0   100.1  2.62            6.94    -1    871          8257      18.26%
previous     0.6    0.0     2.3 41.85         4506.86     0    275          8257      18.26%

additional: 10 numeric columns
        column    mean  median   std  skew excess_kurtosis    min    max  iqr_outliers outlier_pct
           age    40.0    38.0  10.4  0.78            0.79     17     98           469       1.14%
 

## 9. Negative-value checks

Negative values in money or index columns are a common data-quality flag. Each one here is either
explainable or must be reported as unexplained.

In [11]:
NEGATIVE_VERDICTS = {
    "balance": "legitimate overdraft - a negative yearly balance is real, not an error",
    "pdays": "the 'never contacted before' sentinel (section 7), converted to a flag in T12",
    "cons.conf.idx": "the consumer confidence index is negative by construction",
    "emp.var.rate": "employment variation rate can genuinely be negative in a downturn",
}
negative_rows = []
for name in ("full", "additional"):
    for column in RAW[name].select_dtypes(include="number").columns:
        series = RAW[name][column]
        negatives = int((series < 0).sum())
        if negatives:
            negative_rows.append({
                "variant": name,
                "column": column,
                "negative_rows": negatives,
                "pct_of_rows": negatives / len(series),
                "min": series.min(),
                "verdict": NEGATIVE_VERDICTS.get(column, "UNEXPLAINED - investigate"),
            })

negatives = pd.DataFrame(negative_rows)
print(negatives.to_string(index=False,
                          formatters={"pct_of_rows": "{:.2%}".format, "min": "{:g}".format}))
unexplained = negatives[negatives["verdict"].str.startswith("UNEXPLAINED")]
print()
print(f"Negative-valued columns found: {len(negatives)} | unexplained: {len(unexplained)}")
print()
print("Insight 8: every negative value is explainable - overdrafts, the pdays sentinel, and two")
print("           indices that are negative by construction. None is clipped. Clipping balance at 0")
print("           would destroy real signal about clients who are already borrowing, which is a")
print("           genuine credit-risk feature rather than a data error.")

   variant        column  negative_rows pct_of_rows   min                                                                       verdict
      full       balance           3766       8.33% -8019        legitimate overdraft - a negative yearly balance is real, not an error
      full         pdays          36954      81.74%    -1 the 'never contacted before' sentinel (section 7), converted to a flag in T12
additional  emp.var.rate          17191      41.74%  -3.4             employment variation rate can genuinely be negative in a downturn
additional cons.conf.idx          41188     100.00% -50.8                     the consumer confidence index is negative by construction

Negative-valued columns found: 4 | unexplained: 0

Insight 8: every negative value is explainable - overdrafts, the pdays sentinel, and two
           indices that are negative by construction. None is clipped. Clipping balance at 0
           would destroy real signal about clients who are already borrowing, which is

## 10. Quality figures

Saved to `reports/figures/` with the `00_` prefix so they sort ahead of the Phase 4 EDA figures
(named `01_`–`05_`) and cannot collide with them.

In [12]:
saved_figures = []

# Figure 1 - the pdays sentinel spike, which is the clearest structural anomaly in the data.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, name in zip(axes, ("full", "additional")):
    ax.hist(RAW[name]["pdays"], bins=80, color="#4472c4")
    ax.set_yscale("log")
    ax.set_title(f"{name}: pdays distribution (log count)")
    ax.set_xlabel(f"pdays - sentinel is {pdays_sentinels[name]}")
    ax.set_ylabel("rows (log scale)")
fig.suptitle("pdays is bimodal: one huge sentinel bar, then a separate block of real day counts")
fig.tight_layout()
path = config.FIGURES_DIR / "00_quality_pdays_sentinel.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
saved_figures.append(path)

# Figure 2 - skew per numeric column, with the +/-1 rule-of-thumb lines.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, name in zip(axes, ("full", "additional")):
    numeric[name].set_index("column")["skew"].plot(kind="bar", ax=ax, color="#c0504d")
    ax.axhline(1, color="grey", linestyle="--", linewidth=1)
    ax.axhline(-1, color="grey", linestyle="--", linewidth=1)
    ax.set_title(f"{name}: skew of numeric columns")
    ax.set_ylabel("skew (dashed lines at -1 and +1)")
    ax.tick_params(axis="x", labelrotation=90)
fig.tight_layout()
path = config.FIGURES_DIR / "00_quality_numeric_skew.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
saved_figures.append(path)

# Figure 3 - how much of each categorical column is recorded as 'unknown'.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, name in zip(axes, ("full", "additional")):
    shares = unknown[name].set_index("column")["pct_of_rows"]
    shares = shares[shares > 0].sort_values()
    shares.plot(kind="barh", ax=ax, color="#9bbb59")
    ax.set_title(f"{name}: share of rows recorded as 'unknown'")
    ax.set_xlabel("share of rows")
    ax.xaxis.set_major_formatter(lambda value, _position: f"{value:.0%}")
fig.tight_layout()
path = config.FIGURES_DIR / "00_quality_unknown_profile.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
saved_figures.append(path)

for path in saved_figures:
    print(f"saved {path.relative_to(REPO_ROOT)}  ({path.stat().st_size / 1024:.0f} KB)")

saved reports\figures\00_quality_pdays_sentinel.png  (63 KB)
saved reports\figures\00_quality_numeric_skew.png  (82 KB)
saved reports\figures\00_quality_unknown_profile.png  (54 KB)


## 11. Findings

Nine findings, each measured in the section named against it. The numbers come from `data/raw/` as
loaded by `src/data_loader.py`, and section 12 re-establishes them a second time against the T10
data dictionary.

1. **The files are what the plan says, and the target is imbalanced.** Primary set 45,211 × 17;
   additional 41,188 × 21; 10% sample 4,521 × 17. Only 5,289 clients subscribed — **11.70%** — so
   predicting "no" for everybody already scores ≈88.3% accuracy while converting nobody. This is
   the measured justification for excluding accuracy as a headline metric (decision D04).
2. **Missingness is invisible to `isna()`.** Zero NaN cells in the entire dataset, yet eight
   categorical columns across the two variants carry a literal `unknown` level, and `pdays` hides
   non-response inside a sentinel. A null count is not a quality check here.
3. **`unknown` is far too common to be dropped.** `poutcome` is `unknown` for **36,959** classic
   rows (81.7%) and `default` is `unknown` for **8,597** additional rows (20.9%). Listwise deletion
   on that basis would remove the majority of the data, so the value has to be treated as its own
   category — a policy set deliberately in T12.
4. **`contact` loses 28.8% of the classic set** (13,020 rows) to `unknown`, and the additional
   variant removes the level entirely. The two variants do not even carry identical feature
   semantics, which is why D19 requires separately fitted pipelines.
5. **`pdays` is sentinel-encoded and the sentinel differs by variant**: **−1** (36,954 rows) in the
   classic set, **999** (39,673 rows) in the additional set. Both distributions are bimodal, so the
   published means (40.2 and 962.5) are meaningless; real day counts only exist for the 18.3% and
   3.7% of rows that were previously contacted.
6. **The additional set's sentinel does not reliably mean "never contacted" — the classic set's
   does.** `pdays != −1` and `previous > 0` select exactly the same 8,257 rows in the primary set
   (zero contradictions), but **4,110** rows in the additional set carry `999` while `previous` runs
   1–6 and `poutcome` is `failure`. So T12 must derive the `previously_contacted` flag from
   `previous > 0` rather than from the sentinel (decision D21), or it will mislabel 4,110 clients
   and give them a wrong recency value. This is the finding a numeric quality check cannot see.
7. **The additional variant contains 12 exact duplicate rows**; the primary dataset contains none.
   They are 0.03% of rows, left untouched in `data/raw/`, and flagged for an explicit decision in
   T14 rather than silently dropped or silently kept.
8. **No column is constant**, so nothing is dropped for being information-free — but concentration
   is extreme: `default` is **98.2%** "no" in the primary set (only 814 clients are in default) and
   `pdays` is **96.3%** on the sentinel in the additional set. The five economic indicators carry
   only 10–26 distinct values across 41,188 rows, so they behave more like ordered categories than
   continuous measurements, and their importance scores should be read with that in mind.
9. **The money and campaign columns are heavy-tailed, not broken.** `balance` is the most skewed
   (−8,019 to 102,127 — skew ≫ 1), and its negatives are legitimate overdrafts. T13 must therefore
   choose between a robust scaler and winsorising, and justify it, rather than deleting outliers
   that are mostly wealthy clients.

**Two small documentation refinements, reported not applied** (found by section 12, listed there as
NUANCE so they are not lost): the dictionary's `pdays` "other values" cell for the additional set
reads `0 – 999`, which repeats the sentinel inside the column that is supposed to exclude it (the
measured non-sentinel range is **0 – 27**), and the classic row reads `0 – 871` where the measured
non-sentinel minimum is **1**.

**Scope note:** this notebook documents quality; it does not fix anything. Every cleaning decision
(`unknown` policy, sentinel handling, outlier treatment, duplicate handling) belongs to Phase 3 and
is implemented in `src/clean.py` (T12–T14).

## 12. Independent cross-check against `docs/02_data_dictionary.md`

The dictionary was written from its own measurement pass. This section re-measures the same facts
and compares them, so a drift between the two documents cannot go unnoticed.

The comparison is **generic**: it locates the documented figure by its table row label and column
position, parses the number out of the markdown, and compares it with the value measured here. It
never searches for the number it expects, which would make the check circular.

Outcomes:

| Status | Meaning |
|---|---|
| `MATCH` | The dictionary's figure and this notebook's measurement agree. |
| `DISAGREE` | The two differ. Reported loudly; the dictionary is **not** overwritten. |
| `NOT FOUND` | The expected row or cell is absent from the dictionary — itself a defect. |
| `NUANCE` | The number is right but the documented **meaning** is incomplete, checked against an independent column. |
| `SKIPPED` | A documented figure this notebook cannot measure (state why, do not guess). |

This block also earns its keep as a parser test. On the first executed run it reported six
`DISAGREE`s, all six of which turned out to be defects in **this notebook's own parsing** — a rate
cell that captured `5,289` as `5.0`, a count cell that concatenated `36,954` with `81.7` into
`36954817`, and a tuple parser that dropped the minus sign from `−1` — with zero real drift. Both
kinds of failure look identical on screen, so every disagreement below has to be classified before
it is acted on.

In [13]:
DICTIONARY = config.DOCS_DIR / "02_data_dictionary.md"

# The dictionary writes negative sentinels with a Unicode minus sign; normalise it so the parser
# and the comparison see plain ASCII hyphens on both sides.
dictionary_text = DICTIONARY.read_text(encoding="utf-8").replace("\u2212", "-")

# Split on level-2 headings ("## 1. ..." etc.) so each check can be aimed at one section.
SECTIONS = {}
for chunk in re.split(r"\n## ", dictionary_text):
    heading = chunk.split("\n", 1)[0].strip()
    SECTIONS[heading.split(".")[0].strip()] = chunk
print("Dictionary sections parsed:", sorted(SECTIONS))
print(f"Checked against: {DICTIONARY.relative_to(REPO_ROOT)}")
print()


def row_cells(section, needle):
    # Cells of the first markdown table row in `section` containing `needle`.
    for line in SECTIONS[section].splitlines():
        line = line.strip()
        if line.startswith("|") and needle in line:
            return [cell.strip() for cell in line.strip("|").split("|")]
    return []


def first_int(text):
    # The FIRST integer in a cell, sign and thousands separators included. Parsing every digit in
    # the cell instead would turn "**36,954** (81.7% of rows)" into 36954817 - which is exactly the
    # bug this helper was written to fix.
    match = re.search(r"-?[0-9][0-9,]*", text or "")
    return int(match.group(0).replace(",", "")) if match else None


def first_pct(text):
    # The number immediately before a percent sign, for cells like "5,289 (**11.70%**)" where the
    # first number in the cell is the count, not the rate.
    match = re.search(r"-?[0-9]+(?:\.[0-9]+)?\s*%", text or "")
    return float(match.group(0).rstrip("% ")) if match else None


def every_int(text):
    # Every integer in a cell, keeping signs, for cells that hold a tuple of values.
    return tuple(int(number.replace(",", "")) for number in re.findall(r"-?[0-9][0-9,]*", text or ""))


checks = []


def add_check(label, documented, measured, tolerance=0.0, status=None, note=""):
    if status is None:
        if documented is None or measured is None:
            status = "NOT FOUND"
        elif isinstance(documented, float) and tolerance:
            status = "MATCH" if abs(documented - measured) <= tolerance else "DISAGREE"
        else:
            status = "MATCH" if documented == measured else "DISAGREE"
    checks.append({"check": label, "documented": documented, "measured": measured,
                   "status": status, "note": note})


# --- Section 1: the inventory table, one row per dataset ---------------------
for variant, needle in (("full", "`bank-full.csv`"),
                        ("sample", "`bank.csv`"),
                        ("additional", "`bank-additional-full.csv`")):
    cells = row_cells("1", needle)
    df = RAW[variant]
    add_check(f"sec 1: {variant} rows", first_int(cells[2]) if cells else None, len(df))
    add_check(f"sec 1: {variant} columns", first_int(cells[3]) if cells else None, df.shape[1])
    add_check(f"sec 1: {variant} positive rate %",
              first_pct(cells[5]) if cells else None,
              round(100 * float((df["y"] == "yes").mean()), 2), tolerance=0.005)
    add_check(f"sec 1: {variant} NaN cells",
              first_int(cells[6]) if cells else None, int(df.isna().sum().sum()))
    add_check(f"sec 1: {variant} duplicate rows",
              first_int(cells[7]) if cells else None, int(df.duplicated().sum()))

# Documented but not measurable here: the additional 10% sample is not a loader variant.
extra_sample_cells = row_cells("1", "`bank-additional.csv`")
checks.append({
    "check": "sec 1: additional_sample shape",
    "documented": every_int(extra_sample_cells[2] + " " + extra_sample_cells[3]) if extra_sample_cells else None,
    "measured": "not measured",
    "status": "SKIPPED",
    "note": "the additional 10% sample is not exposed as a variant of src/data_loader.py, so this "
            "notebook profiles only the three variants the loader serves",
})

# --- Sections 3-5: which columns are shared, classic-only, additional-only ---
def documented_columns(section):
    pattern = re.compile(r"^\|\s*`([^`]+)`\s*\|")
    names = []
    for line in SECTIONS[section].splitlines():
        match = pattern.match(line.strip())
        if match:
            names.append(match.group(1))
    return names


documented_shared = sorted(documented_columns("3"))
documented_classic_only = sorted(documented_columns("4"))
documented_additional_only = sorted(documented_columns("5"))

full_columns = set(RAW["full"].columns) - {config.TARGET}
additional_columns = set(RAW["additional"].columns) - {config.TARGET}
measured_shared = sorted(full_columns & additional_columns)
measured_classic_only = sorted(full_columns - additional_columns)
measured_additional_only = sorted(additional_columns - full_columns)

for label, documented, measured in (
    ("sec 3: shared column count", documented_shared, measured_shared),
    ("sec 4: classic-only column count", documented_classic_only, measured_classic_only),
    ("sec 5: additional-only column count", documented_additional_only, measured_additional_only),
):
    same = documented == measured
    checks.append({
        "check": label,
        "documented": f"{len(documented)} columns",
        "measured": f"{len(measured)} columns" if same else f"{len(measured)} columns {measured}",
        "status": "MATCH" if same else "DISAGREE",
        "note": "" if same else "the column NAMES differ, not just the counts",
    })

# --- Section 4: ranges of the two classic-only numeric columns ---------------
for column in ("balance", "day"):
    cells = row_cells("4", f"`{column}`")
    add_check(f"sec 4: {column} min/max",
              every_int(cells[2])[:2] if cells else None,
              (int(RAW["full"][column].min()), int(RAW["full"][column].max())))

# --- Section 5.2: distinct-value counts of the economic indicators -----------
for column in ("emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"):
    cells = row_cells("5", f"`{column}`")
    match = re.search(r"\(([0-9,]+) distinct", cells[2]) if cells else None
    documented_distinct = int(match.group(1).replace(",", "")) if match else None
    add_check(f"sec 5.2: {column} distinct values",
              documented_distinct, int(RAW["additional"][column].nunique()))

# --- Section 6.1: the per-column 'unknown' counts ----------------------------
def documented_unknown(cell):
    # The dictionary writes an em dash where the level does not exist in that variant, which as a
    # count is zero.
    return 0 if cell.startswith("\u2014") else first_int(cell)


for column in ("poutcome", "contact", "education", "job", "default", "housing", "loan", "marital"):
    cells = row_cells("6", f"`{column}`")
    if not cells:
        add_check(f"sec 6.1: '{column}' unknown counts", None, None)
        continue
    add_check(f"sec 6.1: '{column}' unknown in classic",
              documented_unknown(cells[1]),
              int((RAW["full"][column] == "unknown").sum()))
    add_check(f"sec 6.1: '{column}' unknown in additional",
              documented_unknown(cells[3]),
              int((RAW["additional"][column] == "unknown").sum()))

# --- Section 6.2: the pdays sentinels, their counts, and what they mean ------
cells = row_cells("6", "`full` (classic)")
sentinel_row_full = cells
add_check("sec 6.2: classic pdays sentinel",
          first_int(sentinel_row_full[1]) if sentinel_row_full else None, pdays_sentinels["full"])
add_check("sec 6.2: classic sentinel row count",
          first_int(sentinel_row_full[3]) if sentinel_row_full else None,
          int((RAW["full"]["pdays"] == pdays_sentinels["full"]).sum()))

cells = row_cells("6", "**999**")
sentinel_row_additional = cells
add_check("sec 6.2: additional pdays sentinel",
          first_int(sentinel_row_additional[1]) if sentinel_row_additional else None,
          pdays_sentinels["additional"])
add_check("sec 6.2: additional sentinel row count",
          first_int(sentinel_row_additional[3]) if sentinel_row_additional else None,
          int((RAW["additional"]["pdays"] == pdays_sentinels["additional"]).sum()))

# Semantic check: the dictionary glosses the additional sentinel as "not previously contacted" and
# reports the sentinel count as that figure. Those are not the same number.
additional_sentinel = pdays_sentinels["additional"]
never_contacted_additional = int(((RAW["additional"]["pdays"] == additional_sentinel)
                                  & (RAW["additional"]["previous"] == 0)).sum())
add_check("sec 6.2: additional sentinel rows that had NO previous contact",
          first_int(sentinel_row_additional[3]) if sentinel_row_additional else None,
          never_contacted_additional,
          status="NUANCE",
          note=f"the dictionary reports every one of the sentinel rows as 'not previously "
               f"contacted'; measured, {int((RAW['additional']['pdays'] == additional_sentinel).sum()) - never_contacted_additional:,} of them have "
               f"previous > 0 (all with poutcome = 'failure'), so the flag must come from "
               f"previous > 0, not from the sentinel - decision D21")

# NUANCE: the "other (real) values" cell must not repeat the sentinel.
full_real = RAW["full"]["pdays"][RAW["full"]["pdays"] != pdays_sentinels["full"]]
add_check("sec 6.2: classic non-sentinel pdays range",
          every_int(sentinel_row_full[4])[:2] if sentinel_row_full else None,
          (int(full_real.min()), int(full_real.max())),
          status="NUANCE",
          note="the lower bound is documented as 0, but no classic row has pdays = 0: the measured "
               "range of real values is 1-871")

additional_real = RAW["additional"]["pdays"][RAW["additional"]["pdays"] != additional_sentinel]
add_check("sec 6.2: additional non-sentinel pdays range",
          every_int(sentinel_row_additional[4])[:2] if sentinel_row_additional else None,
          (int(additional_real.min()), int(additional_real.max())),
          status="NUANCE",
          note="the cell reads 0-999, which repeats the sentinel inside the column that is meant to "
               "exclude it; the measured non-sentinel range is 0-27")

# --- Section 7: the variant comparison table ---------------------------------
cells = row_cells("7", "Rows × columns")
add_check("sec 7: shape of classic vs additional",
          every_int(cells[1] + " " + cells[2]) if cells else None,
          (len(RAW["full"]), RAW["full"].shape[1], len(RAW["additional"]), RAW["additional"].shape[1]))

cells = row_cells("7", "Duplicate rows")
add_check("sec 7: duplicate rows classic vs additional",
          every_int(cells[1] + " " + cells[2]) if cells else None,
          (int(RAW["full"].duplicated().sum()), int(RAW["additional"].duplicated().sum())))

cells = row_cells("7", "Positive rate")
documented_rates = (first_pct(cells[1]), first_pct(cells[2])) if cells else (None, None)
measured_rates = (round(100 * float((RAW["full"]["y"] == "yes").mean()), 2),
                  round(100 * float((RAW["additional"]["y"] == "yes").mean()), 2))
rates_agree = (documented_rates[0] is not None
               and abs(documented_rates[0] - measured_rates[0]) <= 0.005
               and abs(documented_rates[1] - measured_rates[1]) <= 0.005)
checks.append({"check": "sec 7: positive rate % classic vs additional",
               "documented": documented_rates, "measured": measured_rates,
               "status": "MATCH" if rates_agree else ("NOT FOUND" if documented_rates[0] is None else "DISAGREE"),
               "note": ""})

cells = row_cells("7", "`pdays` sentinel")
add_check("sec 7: pdays sentinel classic vs additional",
          every_int(cells[1] + " " + cells[2]) if cells else None,
          (pdays_sentinels["full"], pdays_sentinels["additional"]))

# --- Report ------------------------------------------------------------------
results = pd.DataFrame(checks)
print("Cross-check results (documented value parsed out of the dictionary vs value measured here):")
print(results[["check", "documented", "measured", "status"]].to_string(index=False))
print()

counts_by_status = results["status"].value_counts().to_dict()
print(f"Checks run: {len(results)} | by status: {counts_by_status}")
print()

failures = results[results["status"].isin(["DISAGREE", "NOT FOUND"])]
if failures.empty:
    print("RESULT: no numeric disagreement. Every figure quoted in docs/02_data_dictionary.md was")
    print("        reproduced independently here from data/raw/, and no documented value was")
    print("        written over. The dictionary and this notebook agree.")
else:
    print("RESULT: the two sources are NOT in full agreement. Nothing has been overwritten - the")
    print("        dictionary keeps its values until the difference is resolved deliberately.")
    print("        First decide whether the parser or the data is wrong; on the first executed run")
    print("        of this notebook, all six disagreements were parser defects.")
    for row in failures.itertuples():
        print(f"  [{row.status}] {row.check}")
        print(f"      documented: {row.documented}")
        print(f"      measured  : {row.measured}")
    for label in ("sec 3: shared column count",
                  "sec 4: classic-only column count",
                  "sec 5: additional-only column count"):
        row = failures[failures["check"] == label]
        if not row.empty:
            print()
            print(f"        {label}: the counts differ, so compare the measured lists printed at")
            print("        the end of this cell against the dictionary's own tables in sections 3-5.")

nuances = results[results["status"] == "NUANCE"]
if not nuances.empty:
    print()
    print(f"NUANCE ({len(nuances)}) - the documented NUMBER is right, but the documented MEANING is")
    print("        incomplete. Checked against an independent column, not against the same value:")
    for row in nuances.itertuples():
        print(f"  {row.check}")
        print(f"      documented: {row.documented}   measured: {row.measured}")
        print(f"      {row.note}")
    print("        The dictionary is left as written: T11 reports, it does not overwrite another")
    print("        task's deliverable. The consequence is carried into T12 and decision D21.")

skipped = results[results["status"] == "SKIPPED"]
if not skipped.empty:
    print()
    print(f"SKIPPED ({len(skipped)}) - a documented figure this notebook cannot measure:")
    for row in skipped.itertuples():
        print(f"  {row.check}: documented {row.documented} - {row.note}")

print()
print("The three findings this check exists to protect, re-measured directly:")
print(f"  'balance' present in the additional set? {'balance' in RAW['additional'].columns}")
print(f"  'month'   present in the additional set? {'month' in RAW['additional'].columns}")
print(f"  shared columns ({len(measured_shared)}): {measured_shared}")
print(f"  classic-only ({len(measured_classic_only)}): {measured_classic_only}")
print(f"  additional-only ({len(measured_additional_only)}): {measured_additional_only}")
print("  A wrong classification of these lists was the error T10 caught, so it is re-asserted on")
print("  every run rather than trusted.")

Dictionary sections parsed: ['# DepositIQ — Data Dictionary', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Checked against: docs\02_data_dictionary.md



Cross-check results (documented value parsed out of the dictionary vs value measured here):
                                                         check             documented               measured  status
                                              sec 1: full rows                  45211                  45211   MATCH
                                           sec 1: full columns                     17                     17   MATCH
                                   sec 1: full positive rate %                   11.7                   11.7   MATCH
                                         sec 1: full NaN cells                      0                      0   MATCH
                                    sec 1: full duplicate rows                      0                      0   MATCH
                                            sec 1: sample rows                   4521                   4521   MATCH
                                         sec 1: sample columns                     17    

## 13. Reproducibility record

Everything above is regenerated from `data/raw/` on every run; nothing in this notebook is
hand-typed data. The footer records the environment the results belong to.

In [14]:
print("Notebook   : notebooks/01_data_understanding.ipynb  (T11, rubric criterion 2)")
print("Executed   :", datetime.now().isoformat(timespec="seconds"))
print("Repo root  :", REPO_ROOT)
print("Datasets   :", {name: tuple(frame.shape) for name, frame in RAW.items()})
print("Dictionary :", DICTIONARY.relative_to(REPO_ROOT),
      "| cross-check status:", results["status"].value_counts().to_dict())
print("Figures    :")
for path in sorted(config.FIGURES_DIR.glob("00_quality_*.png")):
    print(f"   {path.name:<40} {path.stat().st_size / 1024:>6.0f} KB")
print()
print("Rerun with : jupyter nbconvert --to notebook --execute --inplace "
      "notebooks/01_data_understanding.ipynb")
print("Next task  : T12 - decide and implement the 'unknown' and sentinel handling in src/clean.py,")
print("             deriving previously_contacted from 'previous > 0' (decision D21).")

Notebook   : notebooks/01_data_understanding.ipynb  (T11, rubric criterion 2)
Executed   : 2026-10-27T18:31:08
Repo root  : D:\diabetes+130-us+hospitals+for+years+1999-2008\depositiq-term-deposit-prediction
Datasets   : {'full': (45211, 17), 'additional': (41188, 21), 'sample': (4521, 17)}
Dictionary : docs\02_data_dictionary.md | cross-check status: {'MATCH': 49, 'NUANCE': 3, 'SKIPPED': 1}
Figures    :
   00_quality_numeric_skew.png                  82 KB
   00_quality_pdays_sentinel.png                63 KB
   00_quality_unknown_profile.png               54 KB

Rerun with : jupyter nbconvert --to notebook --execute --inplace notebooks/01_data_understanding.ipynb
Next task  : T12 - decide and implement the 'unknown' and sentinel handling in src/clean.py,
             deriving previously_contacted from 'previous > 0' (decision D21).
